# Machine Learning - Credit Risk

Notebook inicial para classificacao e regressao com Scikit-Learn.

In [8]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    root_mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

load_dotenv(Path('..') / '.env')
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER', 'postgres')}:{os.getenv('DB_PASSWORD', '')}@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '5432')}/{os.getenv('DB_NAME', 'credit_risk')}"
)

query = '''
SELECT
    f.id_aplicacao,
    c.idade,
    c.renda_anual,
    c.posse_residencia,
    c.tempo_empregada,
    p.finalidade_emprestimo,
    p.classificacao_emprestimo,
    h.inadimplencia_historica,
    h.duracao_historico_credito,
    f.montante_emprestimo,
    f.juros_aplicado,
    f.porcentagem_renda,
    f.status_emprestimo
FROM analytics.fato_emprestimo f
JOIN analytics.dim_cliente c ON c.id_cliente = f.id_cliente
JOIN analytics.dim_perfil_emprestimo p ON p.id_perfil_emprestimo = f.id_perfil_emprestimo
JOIN analytics.dim_historico_credito h ON h.id_historico_credito = f.id_historico_credito
'''

df = pd.read_sql(query, engine)
df.head()

,id_aplicacao,idade,renda_anual,posse_residencia,tempo_empregada,finalidade_emprestimo,classificacao_emprestimo,inadimplencia_historica,duracao_historico_credito,montante_emprestimo,juros_aplicado,porcentagem_renda,status_emprestimo
0,1,22,59000.0,RENT,123.0,PERSONAL,D,Y,3,35000.0,16.02,0.59,1
1,2,21,9600.0,OWN,5.0,EDUCATION,B,N,2,1000.0,11.14,0.10,0
2,3,25,9600.0,MORTGAGE,1.0,MEDICAL,C,N,3,5500.0,12.87,0.57,1
3,4,23,65500.0,RENT,4.0,MEDICAL,C,N,2,35000.0,15.23,0.53,1
4,5,24,54400.0,RENT,8.0,MEDICAL,C,Y,4,35000.0,14.27,0.55,1


In [5]:
target = 'status_emprestimo'
X = df.drop(columns=[target])
y = df[target]

categorical_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object', 'str']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'knn': KNeighborsClassifier(n_neighbors=5),
    'decision_tree': DecisionTreeClassifier(random_state=42),
    'random_forest': RandomForestClassifier(random_state=42),
    'logistic_regression': LogisticRegression(max_iter=1000),
}

for name, model in models.items():
    clf = Pipeline([('preprocessor', preprocessor), ('model', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(name)
    print('accuracy:', accuracy_score(y_test, y_pred))
    print('precision:', precision_score(y_test, y_pred, zero_division=0))
    print('recall:', recall_score(y_test, y_pred, zero_division=0))
    print('f1:', f1_score(y_test, y_pred, zero_division=0))
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))
    print('-' * 40)

knn
accuracy: 0.884149148381157
precision: 0.8338338338338338
recall: 0.5857946554149086
f1: 0.688145394465097
[[4929  166]
 [ 589  833]]
              precision    recall  f1-score   support

           0       0.89      0.97      0.93      5095
           1       0.83      0.59      0.69      1422

    accuracy                           0.88      6517
   macro avg       0.86      0.78      0.81      6517
weighted avg       0.88      0.88      0.88      6517

----------------------------------------
decision_tree
accuracy: 0.8987264078563756
precision: 0.7649513212795549
recall: 0.7735583684950773
f1: 0.7692307692307693
[[4757  338]
 [ 322 1100]]
              precision    recall  f1-score   support

           0       0.94      0.93      0.94      5095
           1       0.76      0.77      0.77      1422

    accuracy                           0.90      6517
   macro avg       0.85      0.85      0.85      6517
weighted avg       0.90      0.90      0.90      6517

-----------------

In [9]:
regression_df = df.dropna(subset=['juros_aplicado']).copy()
X_reg = regression_df.drop(columns=['juros_aplicado'])
y_reg = regression_df['juros_aplicado']

X_reg = X_reg.drop(columns=['status_emprestimo'])
categorical_cols_reg = X_reg.select_dtypes(include=['object', 'str']).columns.tolist()
numeric_cols_reg = X_reg.select_dtypes(exclude=['object', 'str']).columns.tolist()
preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols_reg),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols_reg),
    ]
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
reg_pipeline = Pipeline([('preprocessor', preprocessor_reg), ('model', LinearRegression())])
reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg = reg_pipeline.predict(X_test_reg)

rmse = root_mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
print({'rmse': rmse, 'r2': r2})

{'rmse': 0.9945194973466778, 'r2': 0.9078080811739667}
